# Link the Sample Firms to Their KPSS Patents

| Item | Description |
|---|---|
| Goal | Two steps. First, link every firm of the T3 sample (gvkey) to its CRSP permno(s) through the CRSP-Compustat link table, keeping the link dates. Second, from the KPSS patent data (Kogan, Papanikolaou, Seru, and Stoffman, 2017; extended to 2025), keep the patents filed by a sample firm during the sample period: a patent is attributed to the firm whose permno link is in force on the patent's filing date. |
| Input | `crsp_compustat_link.dta` (the CRSP-Compustat link table, one row per gvkey-permno link with `LINKDT` and `LINKENDDT`) and `Technological-Innovation-Resource-Allocation-and-Growth-Extended-Data/KPSS_2025.zip` (the patent-level KPSS table), both in the section's `Raw` folder on scratch; `output/T3.4.1/filings.csv` from T3 for the sample firms and their fiscal years. |
| Output | `output/X1.1.1/sample_firm_permno_links.csv`: one row per gvkey-permno link kept. `output/X1.1.1/patent_counts_by_firm.csv` and `patent_counts_by_filing_year.csv`: summaries. On scratch, `Processed/sample_firm_patents.parquet` and `.csv`: one row per patent filed by a sample firm in the sample period, with the firm, the dates, the KPSS value, and the citations. |
| Unit | A patent (`patent_num`) in the KPSS table; a firm is a gvkey of the all-years T3 sample (1,058 firms, fiscal 1999 to 2025). |
| Sample period | Calendar years `SAMPLE_START` to `SAMPLE_END`, the first and last fiscal year of the T3 sample (1999 to 2025), applied to the patent's filing date. The flag `filing_year_in_firm_sample` marks patents whose filing year is one of the firm's own sample fiscal years. |
| Links | Links of type `LC` or `LU` (the researched links) with primary flag `P` or `C`, as is standard for the CRSP-Compustat link; a patent is matched to a gvkey when `LINKDT` is at or before and `LINKENDDT` at or after the filing date. A link still in force has a missing `LINKENDDT` in the Stata file (the CRSP code `E`); it is treated as open-ended, written as `OPEN_END_DATE`. Where two links give a patent two gvkeys, the primary link (`P`) wins and the case is counted. |
| Paths | `Raw` sits at the section level on scratch (it was created before this stage folder existed); `Processed` sits under the stage folder as in S5. |

The KPSS data are to be cited as Kogan, L., Papanikolaou, D., Seru, A., and Stoffman, N. (2017), Technological Innovation, Resource Allocation, and Growth, *Quarterly Journal of Economics*, 132(2), 665 to 712, with the extended data from the authors' GitHub repository.


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and the link rules

All folders in one place. The two roots are the project folder on `/project2` and the scratch folder; the T3 filing list is read from T3's `output` folder.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")
NOTEBOOK_SECTION = Path("X1 - Example of Research Enabled by LLM Pipeline")
NOTEBOOK_SUBFOLDER = NOTEBOOK_SECTION / "X1.1 - Privacy Patents and Topic Relevance"
NOTEBOOK_NAME = "T4.1.1"  # the identifier names this notebook's output folder
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SECTION / "Raw"  # section level, see the header
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SECTION / "output" / NOTEBOOK_NAME
INPUT_FOLDER = PROJECT_FOLDER / "T3 - Topic Relevance and Corporate Disclosure/TEMP/output"
LINK_FILE = RAW_FOLDER / "crsp_compustat_link.dta"
KPSS_ZIP = RAW_FOLDER / "Technological-Innovation-Resource-Allocation-and-Growth-Extended-Data/KPSS_2025.zip"
FILINGS_FILE = INPUT_FOLDER / "T3.4.1/filings.csv"

SAMPLE = "all_years"  # the T3 sample: every fiscal year a firm filed
LINK_TYPES = ["LC", "LU"]  # researched links; LS, LX, LD, LN are not used
LINK_PRIMARY = ["P", "C"]  # primary links; J (joint) and N (non-primary) are not used
OPEN_END_DATE = pd.Timestamp("2099-12-31")  # stands in for a missing LINKENDDT, a link still in force
PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Processed: {PROCESSED_FOLDER}")
print(f"Outputs: {OUTPUT_FOLDER}")


Processed: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/T4.1 - Privacy Patents and Topic Relevance/Processed
Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/output/T4.1.1


## Step 3. Load the sample firms and their fiscal years

The T3 filing list, the chosen sample only: one row per firm with its name, its number of filings, its first and last fiscal year, and the set of fiscal years it filed. The sample period is the span of those fiscal years.


In [3]:
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
filings = filings.loc[filings[f"in_{SAMPLE}"].eq(1)]
firms = (filings.groupby("gvkey").agg(conm=("conm", "first"), n_filings=("accessionNo", "size"), first_fyear=("fyear", "min"), last_fyear=("fyear", "max"),
                                      fyears=("fyear", lambda s: frozenset(s))).reset_index())
SAMPLE_START, SAMPLE_END = int(filings["fyear"].min()), int(filings["fyear"].max())
assert firms["gvkey"].is_unique
print(f"Sample firms: {len(firms):,}; filings: {len(filings):,}; sample period: fiscal {SAMPLE_START} to {SAMPLE_END}")
display(firms[["gvkey", "conm", "n_filings", "first_fyear", "last_fyear"]].head(3))


Sample firms: 1,058; filings: 18,779; sample period: fiscal 1999 to 2025


,gvkey,conm,n_filings,first_fyear,last_fyear
0,001013,ADC TELECOMMUNICATIONS INC,12,1999,2010
1,001045,AMERICAN AIRLINES GROUP INC,22,1999,2024
2,001075,PINNACLE WEST CAPITAL CORP,15,1999,2024


## Step 4. Link the sample firms to their permnos

The link table restricted to the sample gvkeys, the researched link types, and the primary flags, with a permno present. A missing end date marks a link still in force and becomes `OPEN_END_DATE`. Every sample firm must get at least one link; a firm can have several permnos over time (share classes, relistings), which the link dates separate. Saved as one row per link.


In [4]:
link = pd.read_stata(LINK_FILE, columns=["GVKEY", "LINKPRIM", "LIID", "LINKTYPE", "LPERMNO", "LPERMCO", "LINKDT", "LINKENDDT"])
link["gvkey"] = link["GVKEY"].astype(str).str.zfill(6)
links = link.loc[link["gvkey"].isin(firms["gvkey"]) & link["LINKTYPE"].isin(LINK_TYPES) & link["LINKPRIM"].isin(LINK_PRIMARY) & link["LPERMNO"].notna()].copy()
links["permno"] = links["LPERMNO"].astype(int)
links["permco"] = links["LPERMCO"].astype(int)
links["link_in_force"] = links["LINKENDDT"].isna()
links["LINKENDDT"] = links["LINKENDDT"].fillna(OPEN_END_DATE)
assert links["LINKDT"].notna().all(), "a link without a start date"
links = links.merge(firms[["gvkey", "conm", "first_fyear", "last_fyear"]], on="gvkey", how="left", validate="many_to_one")
links = links[["gvkey", "conm", "permno", "permco", "LINKTYPE", "LINKPRIM", "LIID", "LINKDT", "LINKENDDT", "link_in_force", "first_fyear", "last_fyear"]].sort_values(["gvkey", "LINKDT"]).reset_index(drop=True)
missing = firms.loc[~firms["gvkey"].isin(links["gvkey"]), ["gvkey", "conm"]]
assert missing.empty, f"{len(missing)} sample firms without a link: {missing['conm'].tolist()[:10]}"
links.to_csv(OUTPUT_FOLDER / "sample_firm_permno_links.csv", index=False, date_format="%Y-%m-%d")
permnos_per_firm = links.groupby("gvkey")["permno"].nunique()
print(f"Links kept: {len(links):,} for {links['gvkey'].nunique():,} firms and {links['permno'].nunique():,} permnos; "
      f"firms with one permno: {int((permnos_per_firm == 1).sum()):,}, with two or more: {int((permnos_per_firm > 1).sum()):,}; "
      f"links still in force (end date missing, set to {OPEN_END_DATE:%Y-%m-%d}): {int(links['link_in_force'].sum()):,}")
print("Link types and primary flags kept:", links.groupby(["LINKTYPE", "LINKPRIM"]).size().to_dict())
display(links.loc[links["gvkey"].isin(permnos_per_firm.index[permnos_per_firm > 1])].head(6))


Links kept: 1,432 for 1,058 firms and 1,196 permnos; firms with one permno: 927, with two or more: 131; links still in force (end date missing, set to 2099-12-31): 665
Link types and primary flags kept: {('LC', 'C'): 172, ('LC', 'P'): 841, ('LU', 'C'): 89, ('LU', 'P'): 330}


,gvkey,conm,permno,permco,LINKTYPE,LINKPRIM,LIID,LINKDT,LINKENDDT,link_in_force,first_fyear,last_fyear
8,001164,MCI INC,11042,61,LC,P,01,1980-11-24,1986-07-28,False,1999,2004
9,001164,MCI INC,11042,61,LC,P,01,1986-08-22,2002-07-29,False,1999,2004
10,001164,MCI INC,90284,45424,LC,P,10,2004-07-14,2006-01-06,False,1999,2004
11,001177,AETNA INC,46850,20026,LC,P,01,1968-09-24,2000-12-13,False,2000,2017
12,001177,AETNA INC,88845,41072,LC,P,01,2000-12-14,2018-11-30,False,2000,2017
40,001567,TRANE INC,10372,22187,LC,C,00X,1950-01-01,1962-01-30,False,1999,2007


## Step 5. Load the KPSS patents filed in the sample period

The patent table straight from the zip. Dates are integers `yyyymmdd`. A few patents have no filing date; they are dropped, and every one of them must have been issued before the sample period so that nothing in the period is lost. Only patents with a filing date in the sample period are kept from here on.


In [5]:
kpss = pd.read_csv(KPSS_ZIP, dtype={"patent_num": "int64", "permno": "int64", "issue_date": "int64", "filing_date": "Int64", "xi_nominal": "float64", "xi_real": "float64", "cites": "int64"})
n_all = len(kpss)
no_filing_date = kpss["filing_date"].isna()
assert (kpss.loc[no_filing_date, "issue_date"] // 10000 < SAMPLE_START).all(), "a patent without a filing date was issued in the sample period"
print(f"Patents without a filing date, all issued before {SAMPLE_START}, dropped: {int(no_filing_date.sum())}")
kpss = kpss.loc[~no_filing_date].reset_index(drop=True)
for column in ["issue_date", "filing_date"]:
    kpss[column] = pd.to_datetime(kpss[column].astype("int64").astype(str), format="%Y%m%d", errors="coerce")
    assert kpss[column].notna().all(), f"{int(kpss[column].isna().sum())} patents with an unreadable {column}"
assert kpss["patent_num"].is_unique, "a patent appears twice in the KPSS table"
kpss["filing_year"] = kpss["filing_date"].dt.year
in_period = kpss.loc[kpss["filing_year"].between(SAMPLE_START, SAMPLE_END)].reset_index(drop=True)
print(f"KPSS patents: {n_all:,}, filed {kpss['filing_date'].min():%Y-%m-%d} to {kpss['filing_date'].max():%Y-%m-%d}, issued to {kpss['issue_date'].max():%Y-%m-%d}; "
      f"{in_period['permno'].nunique():,} permnos")
print(f"Filed in {SAMPLE_START} to {SAMPLE_END}: {len(in_period):,} patents by {in_period['permno'].nunique():,} permnos")
display(in_period.head(3))


Patents without a filing date, all issued before 1999, dropped: 2


KPSS patents: 3,831,908, filed 1834-03-29 to 2025-08-21, issued to 2025-12-30; 5,663 permnos
Filed in 1999 to 2025: 2,348,685 patents by 5,663 permnos


,patent_num,permno,issue_date,filing_date,xi_nominal,xi_real,cites,filing_year
0,5922498,27983,1999-07-13,1999-01-20,15.833909,9.182350,4,1999
1,5929009,18729,1999-07-27,1999-03-05,44.178421,25.619809,7,1999
2,5932534,18729,1999-08-03,1999-03-11,114.255170,66.258494,1,1999


## Step 6. Attach the sample firms to the patents

Every patent of the period is joined to the sample links on permno; a link counts when it is in force on the filing date. A patent that two links give to two gvkeys keeps the primary link; the number of such patents is reported. The flag `filing_year_in_firm_sample` marks patents whose filing year is one of the firm's sample fiscal years (a calendar year against fiscal years, so an approximation for firms whose fiscal year does not end in December).


In [6]:
candidates = in_period.merge(links[["gvkey", "conm", "permno", "permco", "LINKPRIM", "LINKDT", "LINKENDDT"]], on="permno", how="inner")
candidates = candidates.loc[(candidates["LINKDT"] <= candidates["filing_date"]) & (candidates["filing_date"] <= candidates["LINKENDDT"])]
gvkeys_per_patent = candidates.groupby("patent_num")["gvkey"].nunique()
n_ambiguous = int((gvkeys_per_patent > 1).sum())
candidates["primary_rank"] = candidates["LINKPRIM"].map({"P": 0, "C": 1})
patents = (candidates.sort_values(["patent_num", "primary_rank", "LINKDT"]).drop_duplicates("patent_num", keep="first")
           .drop(columns=["primary_rank"]).reset_index(drop=True))
assert patents["patent_num"].is_unique
fyears = firms.set_index("gvkey")["fyears"]
patents["filing_year_in_firm_sample"] = [year in fyears[gvkey] for gvkey, year in zip(patents["gvkey"], patents["filing_year"])]
patents["issue_year"] = patents["issue_date"].dt.year
patents = patents[["patent_num", "gvkey", "conm", "permno", "permco", "filing_date", "filing_year", "issue_date", "issue_year", "xi_nominal", "xi_real", "cites",
                   "filing_year_in_firm_sample", "LINKPRIM", "LINKDT", "LINKENDDT"]].sort_values(["gvkey", "filing_date", "patent_num"]).reset_index(drop=True)
print(f"Patents filed by sample firms in {SAMPLE_START} to {SAMPLE_END}: {len(patents):,} ({len(patents) / len(in_period):.1%} of the period's patents), "
      f"by {patents['gvkey'].nunique():,} of the {len(firms):,} sample firms; {n_ambiguous:,} patents matched two gvkeys and kept the primary link")
print(f"Patents whose filing year is one of the firm's sample fiscal years: {int(patents['filing_year_in_firm_sample'].sum()):,} "
      f"({patents['filing_year_in_firm_sample'].mean():.1%})")
display(patents.head(3))


Patents filed by sample firms in 1999 to 2025: 1,315,652 (56.0% of the period's patents), by 719 of the 1,058 sample firms; 0 patents matched two gvkeys and kept the primary link
Patents whose filing year is one of the firm's sample fiscal years: 1,236,059 (94.0%)


,patent_num,gvkey,conm,permno,permco,filing_date,filing_year,issue_date,issue_year,xi_nominal,xi_real,cites,filing_year_in_firm_sample,LINKPRIM,LINKDT,LINKENDDT
0,6328608,001013,ADC TELECOMMUNICATIONS INC,50906,2902,1999-01-06,1999,2001-12-11,2001,30.395426,16.788872,64,True,P,1979-03-16,2010-12-31
1,6381270,001013,ADC TELECOMMUNICATIONS INC,50906,2902,1999-01-07,1999,2002-04-30,2002,24.815115,13.388365,9,True,P,1979-03-16,2010-12-31
2,6468112,001013,ADC TELECOMMUNICATIONS INC,50906,2902,1999-01-11,1999,2002-10-22,2002,4.207601,2.270104,158,True,P,1979-03-16,2010-12-31


## Step 7. Save the patent sample and the summaries

The patent table goes to `Processed` on scratch as Parquet and CSV; the two summaries, one row per firm and one row per filing year, go to the notebook's `output` folder. Firms without a patent appear in the firm summary with zero patents.


In [7]:
patents.to_parquet(PROCESSED_FOLDER / "sample_firm_patents.parquet", index=False)
patents.to_csv(PROCESSED_FOLDER / "sample_firm_patents.csv", index=False, date_format="%Y-%m-%d", float_format="%.6f")
by_firm = (patents.groupby("gvkey").agg(n_patents=("patent_num", "size"), n_patents_in_firm_years=("filing_year_in_firm_sample", "sum"), first_filing_year=("filing_year", "min"),
                                        last_filing_year=("filing_year", "max"), n_permnos=("permno", "nunique"), sum_xi_real=("xi_real", "sum"), sum_cites=("cites", "sum")))
by_firm = firms[["gvkey", "conm", "n_filings", "first_fyear", "last_fyear"]].merge(by_firm, on="gvkey", how="left")
for column in ["n_patents", "n_patents_in_firm_years", "n_permnos", "sum_cites"]:
    by_firm[column] = by_firm[column].fillna(0).astype(int)
by_firm["sum_xi_real"] = by_firm["sum_xi_real"].fillna(0.0)
by_firm = by_firm.sort_values("n_patents", ascending=False).reset_index(drop=True)
by_firm.to_csv(OUTPUT_FOLDER / "patent_counts_by_firm.csv", index=False, float_format="%.4f")
by_year = patents.groupby("filing_year").agg(n_patents=("patent_num", "size"), n_firms=("gvkey", "nunique"), sum_xi_real=("xi_real", "sum")).reset_index()
by_year.to_csv(OUTPUT_FOLDER / "patent_counts_by_filing_year.csv", index=False, float_format="%.4f")
print(f"Saved: {PROCESSED_FOLDER / 'sample_firm_patents.parquet'} and .csv ({len(patents):,} rows)")
print(f"Saved: {OUTPUT_FOLDER / 'patent_counts_by_firm.csv'} ({len(by_firm):,} firms, {int((by_firm['n_patents'] == 0).sum()):,} without a patent) and patent_counts_by_filing_year.csv ({len(by_year)} years)")
print("Ten firms with the most patents:")
display(by_firm.head(10))
print("Patents by filing year:")
display(by_year.set_index("filing_year").T)


Saved: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/T4.1 - Privacy Patents and Topic Relevance/Processed/sample_firm_patents.parquet and .csv (1,315,652 rows)
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/output/T4.1.1/patent_counts_by_firm.csv (1,058 firms, 339 without a patent) and patent_counts_by_filing_year.csv (27 years)
Ten firms with the most patents:


,gvkey,conm,n_filings,first_fyear,last_fyear,n_patents,n_patents_in_firm_years,first_filing_year,last_filing_year,n_permnos,sum_xi_real,sum_cites
0,006066,INTL BUSINESS MACHINES CORP,27,1999,2025,136020,136020,1999.0,2025.0,1,3.611611e+05,999647
1,006008,INTEL CORP,25,1999,2023,60852,60669,1999.0,2025.0,1,5.136271e+05,373467
2,012141,MICROSOFT CORP,27,1999,2025,48723,48723,1999.0,2024.0,1,1.679439e+06,558967
3,024800,QUALCOMM INC,27,1999,2025,42137,42137,1999.0,2024.0,1,3.084816e+05,214559
4,005047,GE AEROSPACE,15,1999,2013,36289,23564,1999.0,2025.0,1,6.031125e+05,296551
5,007343,MICRON TECHNOLOGY INC,27,1999,2025,34829,34829,1999.0,2025.0,1,1.645878e+05,288562
6,160329,ALPHABET INC,11,2015,2025,34350,19695,2004.0,2025.0,1,6.511490e+05,273434
7,005606,HP INC,27,1999,2025,34039,34039,1999.0,2024.0,1,2.029003e+05,290705
8,001690,APPLE INC,27,1999,2025,33942,33942,1999.0,2025.0,1,1.998104e+06,271556
9,008358,PARKER-HANNIFIN CORP,27,1999,2025,30158,30158,1999.0,2025.0,1,5.141725e+04,150693


Patents by filing year:


filing_year,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
n_patents,3.743000e+04,42273.000000,44696.000000,46348.000000,45959.000000,43504.000000,45349.000000,44833.000000,48263.000000,50082.000000,...,6.807100e+04,6.961400e+04,6.694100e+04,7.024700e+04,6.210900e+04,5.580300e+04,4.266600e+04,2.660500e+04,7472.000000,219.00000
n_firms,3.930000e+02,399.000000,407.000000,416.000000,401.000000,402.000000,384.000000,381.000000,389.000000,387.000000,...,3.600000e+02,3.670000e+02,3.560000e+02,3.500000e+02,3.500000e+02,3.380000e+02,3.270000e+02,3.140000e+02,263.000000,69.00000
sum_xi_real,1.317259e+06,963674.592486,804465.985392,779297.962159,790470.010946,740018.664373,757923.951716,696212.947926,665749.451216,598922.408274,...,1.345920e+06,1.651984e+06,2.036035e+06,2.511847e+06,2.384340e+06,2.346248e+06,2.432083e+06,1.794683e+06,549565.506234,23031.94296
